In [1]:
%pip install -r ../requirements.txt
print('Dependencies installed or already satisfied for Task 3 notebook.')

     ---------------------------------------- 0.0/61.0 kB ? eta -:--:--
     ------ --------------------------------- 10.2/61.0 kB ? eta -:--:--
     ------------------- ------------------ 30.7/61.0 kB 660.6 kB/s eta 0:00:01
     ------------------- ------------------ 30.7/61.0 kB 660.6 kB/s eta 0:00:01
     ------------------------------- ------ 51.2/61.0 kB 262.6 kB/s eta 0:00:01
     -------------------------------------- 61.0/61.0 kB 297.1 kB/s eta 0:00:00
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
     ---------------------------------------- 0.0/137.6 kB ? eta -:--:--
     -------- ---------------------------- 30.7/137.6 kB 660.6 kB/s eta 0:00:01
     ---------------- -------------------- 61.4/137.6 kB 656.4 kB/s eta 0:00:01
     ------------------------ ------------ 92.2/137.6 kB 655.4 kB/s eta 0:00:01
     ------------------------ ------------ 92.2/137.6 kB 655.4 kB/s eta 0:00:01
     ------------------------ ------------ 92.2/137.6 kB 655.4 kB/s eta 0:00:

ERROR: THESE PACKAGES DO NOT MATCH THE HASHES FROM THE REQUIREMENTS FILE. If you have updated the package versions, please update the hashes. Otherwise, examine the package contents carefully; someone may have tampered with them.
    unknown package:
        Expected sha256 3773002da767f0a9323ba1a9b9b5d00d6257dbd2a93107233167cfb581f64717
             Got        61aecc47ab4cff496f8684ad312e98cb98477c7a7a9bffcf5580f4f0927da2db


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# Task 3: RAG Pipeline Construction & Qualitative Evaluation
**CrediTrust Financial — RAG Complaint Chatbot**

This notebook:
1. Loads the pre-built full-scale vector store
2. Demonstrates the retriever and generator in isolation
3. Runs 10 representative questions through the full RAG pipeline
4. Produces a structured evaluation table

## 1. Setup

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Change these paths to match your environment
STORE_PATH = "../vector_store/"
LLM_MODEL  = os.getenv("HF_MODEL", "mistralai/Mistral-7B-Instruct-v0.2")

plt.rcParams["figure.dpi"] = 110

print("RAG Pipeline Evaluation Notebook")
print(f"  Vector store : {STORE_PATH}")
print(f"  LLM model    : {LLM_MODEL}")

ModuleNotFoundError: No module named 'pandas'

## 2. Initialise the RAG Pipeline

In [ ]:
from src.rag_pipeline import RAGPipeline, EVAL_QUESTIONS

pipeline = RAGPipeline(store_path=STORE_PATH, llm_model=LLM_MODEL, top_k=5)
print("Pipeline initialised ✅")

## 3. Component Walkthrough

### 3.1 Retriever — example query

In [ ]:
from src.rag_pipeline import EmbeddingEncoder, VectorStore

encoder = EmbeddingEncoder()
store   = VectorStore(STORE_PATH)

query = "Why are credit card customers complaining about billing?"
vec   = encoder.encode(query)
chunks = store.search(vec, k=5)

print(f"Query: '{query}'\n")
for i, c in enumerate(chunks, 1):
    print(f"[{i}]  {c.product_category:<18}  dist={c.distance:.4f}  |  {c.issue}")
    print(f"     {c.text[:200]}\n")

### 3.2 Prompt Template

In [ ]:
from src.rag_pipeline import build_prompt, SYSTEM_PROMPT

prompt = build_prompt(query, chunks)
print("─── SYSTEM PROMPT ───────────────────────────────────")
print(SYSTEM_PROMPT)
print("\n─── USER PROMPT (first 600 chars) ───────────────────")
print(prompt[:600], "…")

## 4. Full Pipeline — Single Query Demo

In [ ]:
response = pipeline.answer("What are the most common issues with personal loans?", product_filter="Personal Loan")

print("QUESTION:", response.question)
print("\nANSWER:")
print(response.answer)
print(f"\nModel: {response.model_used}")
print(f"Sources retrieved: {len(response.sources)}")

## 5. Qualitative Evaluation

We evaluate 10 representative questions that a Product Manager or Compliance
analyst at CrediTrust might ask.  For each question we record:
- The generated answer (truncated)
- The top retrieved source (product category + issue)
- A quality score 1–5 and brief comments

In [ ]:
results = []

for question, product_filter in EVAL_QUESTIONS:
    print(f"Running: {question}")
    resp = pipeline.answer(question, product_filter=product_filter)
    results.append({
        "question":        question,
        "product_filter":  product_filter or "All",
        "answer":          resp.answer,
        "top_product":     resp.sources[0].product_category if resp.sources else "—",
        "top_issue":       resp.sources[0].issue[:50] if resp.sources else "—",
        "n_sources":       len(resp.sources),
    })

print("\n✅  All queries complete.")

## 6. Evaluation Table

In [ ]:
eval_df = pd.DataFrame(results)

display_df = eval_df.copy()
display_df["answer"] = display_df["answer"].str[:150] + "…"
display_df

### 6.1 Retrieval Summary Plot

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
source_counts = eval_df["top_product"].value_counts().reset_index()
source_counts.columns = ["top_product", "count"]

sns.barplot(data=source_counts, x="top_product", y="count", palette="viridis", ax=ax)
ax.set_title("Top Retrieved Product Category Across Evaluation Questions")
ax.set_xlabel("Product Category")
ax.set_ylabel("Count")
ax.bar_label(ax.containers[0], fmt="%d", padding=3)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

### 6.2 Markdown Table for Report

In [ ]:
print("| # | Question | Answer (excerpt) | Top Source | Top Issue | Score | Comments |")
print("|---|----------|-----------------|------------|-----------|-------|---------|")
for i, row in eval_df.iterrows():
    q  = row["question"]
    a  = row["answer"][:100].replace("|", "\\|") + "…"
    pr = row["top_product"]
    is_ = row["top_issue"]
    print(f"| {i+1} | {q} | {a} | {pr} | {is_} | __ | __ |")

### Scoring Criteria

| Score | Meaning |
|-------|---------|
| 5 | Accurate, well-synthesised, evidence-backed answer with no hallucinations |
| 4 | Mostly correct, minor omissions or slight vagueness |
| 3 | Partially correct; answers the question but misses key points |
| 2 | Superficial or largely irrelevant despite correct retrieval |
| 1 | Wrong or no useful answer |

## 7. Save Evaluation Results

In [ ]:
os.makedirs("../data/processed", exist_ok=True)
eval_df.to_csv("../data/processed/rag_evaluation.csv", index=False)
print("✅  Evaluation saved → data/processed/rag_evaluation.csv")